# 📘 Bài 5 — OOP & tổ chức code

**Tuần 4**

> Là AI Engineer bạn sẽ **dùng class nhiều hơn viết class**. Mục tiêu bài này là đọc hiểu code thư viện và tổ chức code của mình — không cần học sâu về đa kế thừa hay metaclass.

## 1. Class đầu tiên

In [ ]:
class TaiKhoan:
    def __init__(self, chu_tk: str, so_du: float = 0):
        self.chu_tk = chu_tk        # thuộc tính
        self.so_du = so_du
        self.lich_su: list[tuple[str, float]] = []

    def nap(self, tien: float) -> None:      # phương thức
        if tien <= 0:
            raise ValueError(f"So tien phai duong, nhan: {tien}")
        self.so_du += tien
        self.lich_su.append(("nap", tien))

    def __repr__(self) -> str:
        return f"TaiKhoan({self.chu_tk!r}, {self.so_du})"


tk = TaiKhoan("An", 1000)
tk.nap(500)
print(tk)
print(tk.so_du, tk.lich_su)

### `self` là gì?

Khi bạn viết `tk.nap(500)`, Python thực chất gọi `TaiKhoan.nap(tk, 500)` — đối tượng được truyền vào tham số đầu tiên. Tên `self` chỉ là quy ước (rất mạnh, đừng đổi).

In [ ]:
# Chứng minh: gọi qua class, truyền đối tượng thủ công
tk2 = TaiKhoan("Bình", 100)
TaiKhoan.nap(tk2, 50)
print(tk2)

## 2. Mỗi đối tượng có dữ liệu riêng

In [ ]:
a = TaiKhoan("A", 100)
b = TaiKhoan("B", 100)
a.nap(999)

print(a)
print(b)      # b không bị ảnh hưởng
print("a.lich_su:", a.lich_su)
print("b.lich_su:", b.lich_su)

## 3. `__str__` và `__repr__`

In [ ]:
class SanPham:
    def __init__(self, ten, gia):
        self.ten, self.gia = ten, gia

    def __repr__(self):
        return f"SanPham({self.ten!r}, {self.gia})"     # cho lập trình viên

    def __str__(self):
        return f"{self.ten} - {self.gia:,.0f} VND"      # cho người dùng


sp = SanPham("Laptop", 25_000_000)
print(sp)               # dùng __str__
print(repr(sp))         # dùng __repr__
print([sp])             # trong list -> dùng __repr__

> Nếu chỉ viết một cái, hãy viết `__repr__` — Python sẽ dùng nó thay cho `__str__`. Không có cả hai thì bạn nhận được `<__main__.SanPham object at 0x...>`, vô dụng khi debug.

## 4. dataclass — viết class chứa dữ liệu cho gọn

In [ ]:
from dataclasses import dataclass, field


@dataclass
class HocVien:
    ten: str
    tuoi: int = 0
    diem: list[float] = field(default_factory=list)     # ✅ đúng cách


hv = HocVien("An", 25)
print(hv)                          # __repr__ tự sinh
print(hv == HocVien("An", 25))     # __eq__ tự sinh -> so sánh theo giá trị

In [ ]:
# @loi-co-y  <- ô này CỐ Ý lỗi, dùng để bạn xem thông báo lỗi
# ⚠️ Viết list trực tiếp làm mặc định -> dataclass CHẶN NGAY
@dataclass
class Sai:
    diem: list = []

`ValueError: mutable default <class 'list'> for field diem is not allowed`

Chính là bẫy *mutable default argument* ở bài 4 — dataclass chủ động chặn để bạn không tự bắn vào chân mình. Nếu cho phép, **mọi** học viên sẽ dùng chung một list điểm.

> Ở Phase 07 bạn sẽ gặp **Pydantic** — hiểu đơn giản là "dataclass có kiểm tra kiểu dữ liệu thật sự". Đó là công cụ để ép LLM trả JSON đúng cấu trúc.

## 5. Khi nào dùng class, khi nào dùng hàm?

In [ ]:
# ❌ Class chỉ có __init__ và một method -> nên là hàm
class MayTinhThue:
    def __init__(self, thu_nhap):
        self.thu_nhap = thu_nhap

    def tinh(self):
        return self.thu_nhap * 0.1


# ✅ Viết thành hàm, gọn và rõ hơn
def tinh_thue(thu_nhap: float) -> float:
    return thu_nhap * 0.1


print(MayTinhThue(1000).tinh(), tinh_thue(1000))

| Dùng **hàm** khi | Dùng **class** khi |
|---|---|
| Chỉ biến đổi dữ liệu vào → ra | Cần giữ **trạng thái** giữa các lần gọi |
| Không cần nhớ gì | Dữ liệu và hành vi gắn chặt nhau |
| ~90% trường hợp | Ví dụ: kết nối DB, client API, model đã train |

Ví dụ class **đáng dùng** trong công việc AI: một `RAGPipeline` giữ sẵn kết nối vector DB và cấu hình, để mỗi lần hỏi không phải khởi tạo lại.

## 6. Kế thừa — biết là đủ

In [ ]:
class Nguoi:
    def __init__(self, ten):
        self.ten = ten

    def gioi_thieu(self):
        return f"Toi la {self.ten}"


class HocSinh(Nguoi):
    def __init__(self, ten, truong):
        super().__init__(ten)       # gọi __init__ của lớp cha
        self.truong = truong

    def gioi_thieu(self):           # ghi đè phương thức cha
        return f"{super().gioi_thieu()}, hoc tai {self.truong}"


print(Nguoi("An").gioi_thieu())
print(HocSinh("Binh", "ABC").gioi_thieu())
print(isinstance(HocSinh("B", "X"), Nguoi))

> Trong thực tế, kế thừa bị lạm dụng rất nhiều. Nguyên tắc hiện đại: **ưu tiên kết hợp (composition) hơn kế thừa**. Bạn sẽ chủ yếu gặp kế thừa khi *dùng* thư viện: `class MyModel(nn.Module)` trong PyTorch chẳng hạn.

## 7. Module & `if __name__`

In [ ]:
print("Tên module hiện tại:", __name__)

# Trong file .py chạy trực tiếp, __name__ == "__main__"
# Khi file bị import từ nơi khác, __name__ == tên file
#
# Vì vậy mẫu này rất phổ biến:
#
#     if __name__ == "__main__":
#         main()
#
# -> code chỉ chạy khi gọi `python file.py`,
#    KHÔNG chạy khi file bị import.

### Tổ chức thư mục dự án

```
du_an/
├── main.py              # điểm khởi chạy
├── xu_ly_du_lieu.py     # các hàm xử lý
├── cau_hinh.py          # hằng số, thiết lập
└── tests/
    └── test_xu_ly.py
```

```python
# main.py
from xu_ly_du_lieu import doc_csv, lam_sach
from cau_hinh import DUONG_DAN_DATA

if __name__ == "__main__":
    df = doc_csv(DUONG_DAN_DATA)
```

## ✍️ Bài tập

```bash
pytest tests/phase01/test_ex06.py -v
```